# Pipeline ANVISA - Download (Stage 1.0)

Notebook de referencia para manutencao da etapa de download e consolidacao bruta da base ANVISA.

Fluxo coberto aqui (encapsulado em execucao unica):
1. Preparar pastas
2. Raspar links da ANVISA
3. Filtrar periodo de coleta
4. Baixar arquivos
5. Limpar arquivos baixados
6. Consolidar CSV bruto final

Use os parametros da celula 4 e execute a celula 5 para rodar tudo de ponta a ponta.


In [1]:
import os
import sys
import shutil
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pipelines").exists():
    raise RuntimeError("Abra este notebook na raiz do projeto (onde existe a pasta 'pipelines').")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

print(f"Projeto: {PROJECT_ROOT}")


Projeto: c:\Users\luciano\Desktop\Works\Pipeline_Anvisa


In [2]:
import os
import re
import glob
import time
import shutil
import logging
import importlib.util
import concurrent.futures
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag, NavigableString
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm import tqdm

from pipelines.anvisa_base.config_anvisa import (
    ANO_INICIO as CONF_ANO_INICIO,
    MES_INICIO as CONF_MES_INICIO,
    ANO_FIM as CONF_ANO_FIM,
    MES_FIM as CONF_MES_FIM,
    URL_ANVISA,
    MAX_DOWNLOAD_WORKERS,
    MAX_CLEANING_THREADS,
    DOWNLOAD_THREAD_MULTIPLIER,
    DOWNLOAD_MAX_THREADS,
    DOWNLOAD_CHUNK_SIZE,
    DOWNLOAD_RETRIES,
    DOWNLOAD_BACKOFF_SECONDS,
    PASTA_DOWNLOADS_BRUTOS,
    PASTA_ARQUIVOS_LIMPOS,
    ARQUIVO_CONSOLIDADO_TEMP,
)

if not logging.getLogger().handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

HAS_XLRD = importlib.util.find_spec("xlrd") is not None

def scrape_anvisa_links():
    """Raspa a pagina da Anvisa para encontrar os links dos arquivos de precos."""
    logging.info(f"Acessando {URL_ANVISA} para extrair links...")

    meses_map = {
        "janeiro": 1, "fevereiro": 2, "marco": 3, "abril": 4, "maio": 5, "junho": 6,
        "julho": 7, "agosto": 8, "setembro": 9, "outubro": 10, "novembro": 11, "dezembro": 12,
    }
    rx_mesctx = re.compile(
        r"\b(janeiro|fevereiro|mar[çc]o|abril|maio|junho|julho|agosto|setembro|outubro|novembro|dezembro)\s*/\s*(\d{2,4})\b",
        re.IGNORECASE,
    )
    rx_full = re.compile(r"(\d{4})(\d{2})(\d{2})")
    rx_mid = re.compile(r"(\d{4})_(\d{2})_")
    rx_short = re.compile(r"(\d{4})(\d{2})_")

    def normalize_year(y: str) -> int:
        return int(y) if len(y) == 4 else 2000 + int(y)

    def month_name(idx: int) -> str:
        names = [
            "janeiro", "fevereiro", "marco", "abril", "maio", "junho",
            "julho", "agosto", "setembro", "outubro", "novembro", "dezembro",
        ]
        return names[idx - 1]

    soup = BeautifulSoup(requests.get(URL_ANVISA, timeout=30).content, "html.parser")
    core = soup.find(id="content-core")
    if core is None:
        raise RuntimeError("div#content-core nao encontrada na pagina da Anvisa!")

    dados = []
    ctx_year = None
    ctx_month = None

    for node in core.descendants:
        if isinstance(node, NavigableString):
            m = rx_mesctx.search(node.strip().lower())
            if m:
                mes_txt = m.group(1).lower().replace("ç", "c")
                if mes_txt == "marco":
                    ctx_month = 3
                else:
                    ctx_month = meses_map.get(mes_txt)
                ctx_year = normalize_year(m.group(2))
            continue

        if not (isinstance(node, Tag) and node.name == "a"):
            continue

        href = node.get("href", "").strip()
        if not href or "_reso_" in href.lower():
            continue

        txt_upper = node.get_text(" ", strip=True).upper()
        if "XLS" not in txt_upper:
            continue

        href_l = href.lower()
        if "xls_conformidade_gov" not in href_l:
            if not href_l.endswith("json-file-1") or not href_l.split("/")[-1].startswith("5"):
                continue

        ano = None
        mes = None
        for rx in (rx_full, rx_mid, rx_short):
            mm = rx.search(href)
            if mm:
                ano, mes = int(mm.group(1)), int(mm.group(2))
                break

        if not (ano and mes):
            ano, mes = ctx_year, ctx_month

        if ano and mes:
            dados.append({
                "ano": ano,
                "mes": mes,
                "mes_nome": month_name(mes),
                "url": href,
            })

    if not dados:
        return pd.DataFrame(columns=["ano", "mes", "mes_nome", "url"])

    df_links = (
        pd.DataFrame(dados)
        .sort_values(["ano", "mes"])
        .drop_duplicates(["ano", "mes"])
    )
    logging.info(f"Total de links capturados: {len(df_links)}")
    return df_links

def download_files(df_to_download):
    """Baixa os arquivos de uma lista de links em paralelo."""
    workers_download = min(
        MAX_DOWNLOAD_WORKERS * DOWNLOAD_THREAD_MULTIPLIER,
        DOWNLOAD_MAX_THREADS,
    )

    session = requests.Session()
    session.headers.update({"User-Agent": "PipelineAnvisa/1.0"})

    retry = Retry(
        total=DOWNLOAD_RETRIES,
        backoff_factor=DOWNLOAD_BACKOFF_SECONDS,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
    )
    adapter = HTTPAdapter(
        max_retries=retry,
        pool_connections=workers_download,
        pool_maxsize=workers_download,
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    base_folder = Path(PASTA_DOWNLOADS_BRUTOS)
    base_folder.mkdir(exist_ok=True)

    def resolver_destino(row) -> Path:
        ano_cal, mes_cal = int(row.ano), int(row.mes)
        ano_fiscal = ano_cal - 1 if mes_cal <= 3 else ano_cal
        pasta = base_folder / f"anvisa_ano_fiscal_{ano_fiscal}"
        pasta.mkdir(parents=True, exist_ok=True)
        ext = Path(row.url).suffix or ".xls"
        nome = f"{ano_cal}_{mes_cal:02d}_{row.mes_nome}{ext}"
        return pasta / nome

    linhas = [row for _, row in df_to_download.iterrows()]
    pendentes = []
    ja_existentes = 0

    for row in linhas:
        destino = resolver_destino(row)
        if destino.exists():
            ja_existentes += 1
        else:
            pendentes.append(row)

    if ja_existentes:
        logging.info(f"Arquivos ja disponiveis localmente (skip): {ja_existentes}")
    if not pendentes:
        logging.info("Nenhum arquivo novo para download.")
        return

    def download_row(row):
        dest = resolver_destino(row)
        for attempt in range(DOWNLOAD_RETRIES):
            try:
                r = session.get(row.url, stream=True, timeout=60)
                r.raise_for_status()
                with open(dest, "wb") as f:
                    for chunk in r.iter_content(DOWNLOAD_CHUNK_SIZE):
                        f.write(chunk)
                return f"ok ({attempt + 1}): {dest.relative_to(base_folder)}"
            except requests.RequestException:
                time.sleep(DOWNLOAD_BACKOFF_SECONDS)
        return f"falhou: {row.url.split('/')[-1]}"

    workers_ativos = min(workers_download, len(pendentes))
    logging.info(
        f"Iniciando downloads em {workers_ativos} threads ({len(pendentes)} arquivos novos)..."
    )
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers_ativos) as exe:
        resultados = list(
            tqdm(
                exe.map(download_row, pendentes),
                total=len(pendentes),
                desc="Baixando arquivos",
                ncols=100,
            )
        )

    ok = sum(r.startswith("ok") for r in resultados)
    fail = len(resultados) - ok
    logging.info(f"Resumo do download - Sucesso: {ok} | Falha: {fail}")
    if fail:
        for r in resultados:
            if r.startswith("falhou"):
                logging.error(f" - {r}")

def clean_downloaded_files(source_folder, target_folder):
    """Limpa e padroniza os arquivos Excel baixados em paralelo."""
    all_files = sorted(glob.glob(f"{source_folder}/anvisa_ano_fiscal_*/*.xls*"))
    if not all_files:
        logging.warning("Nenhum arquivo .xls/.xlsx encontrado para processar.")
        return

    target_columns = ["PRINCIPIO ATIVO", "SUBSTANCIA", "CNPJ"]

    def detect_excel_container(file_path):
        with open(file_path, "rb") as f:
            header_bytes = f.read(4096)

        stripped = header_bytes.lstrip().lower()
        if stripped.startswith(b"<!doctype") or stripped.startswith(b"<html"):
            return "html"
        if header_bytes.startswith(b"PK\x03\x04"):
            return "zip"
        if header_bytes.startswith(b"\xD0\xCF\x11\xE0"):
            return "ole"
        return "unknown"

    def process_single_file(file_path):
        try:
            filename = os.path.basename(file_path)
            output_name = None
            output_path = None

            try:
                ano_ref, mes_ref = int(filename.split("_")[0]), int(filename.split("_")[1])
                output_name = f"ANVISA_LIMPO_{ano_ref}_{mes_ref:02d}.csv"
                output_path = os.path.join(target_folder, output_name)
                if os.path.exists(output_path) and os.path.getmtime(output_path) >= os.path.getmtime(file_path):
                    return f"SKIP: {filename} -> {output_name} (ja processado)"
            except Exception:
                pass

            file_kind = detect_excel_container(file_path)
            if file_kind == "html":
                return f"ERRO: {file_path} -> Arquivo HTML disfarcado de Excel (download invalido)"

            ext = os.path.splitext(file_path)[1].lower()

            if file_kind == "zip":
                engines_to_try = ["openpyxl"]
            elif file_kind == "ole":
                if not HAS_XLRD:
                    return (
                        f"ERRO: {file_path} -> Arquivo XLS binario detectado, "
                        "mas o pacote 'xlrd' nao esta instalado no ambiente."
                    )
                engines_to_try = ["xlrd"]
            elif ext == ".xlsx":
                engines_to_try = ["openpyxl"]
            elif ext == ".xls":
                engines_to_try = ["xlrd", "openpyxl"] if HAS_XLRD else ["openpyxl"]
            else:
                engines_to_try = ["openpyxl"]

            df_preview = None
            engine_used = None
            errors_by_engine = []

            for engine in engines_to_try:
                try:
                    df_preview = pd.read_excel(file_path, header=None, nrows=200, dtype=str, engine=engine)
                    engine_used = engine
                    break
                except Exception as exc:
                    errors_by_engine.append(f"{engine}: {exc}")
                    continue

            if df_preview is None:
                joined_errors = " | ".join(errors_by_engine) if errors_by_engine else "sem detalhes"
                return f"ERRO: {file_path} -> Nenhum engine funcionou. Erros: {joined_errors}"

            header_row_index = None
            for i, row in df_preview.iterrows():
                row_values = {str(v).strip().upper() for v in row.dropna()}
                if any(col in row_values for col in target_columns):
                    header_row_index = i
                    break

            if header_row_index is None and len(df_preview) > 0:
                for i, row in df_preview.iterrows():
                    non_null_count = row.notna().sum()
                    if non_null_count >= 5:
                        header_row_index = i
                        break

            if header_row_index is None:
                return f"AVISO: Cabecalho nao encontrado -> {file_path}"

            df = pd.read_excel(
                file_path,
                header=None,
                skiprows=header_row_index + 1,
                dtype=str,
                engine=engine_used,
            )
            header = (
                df_preview.iloc[header_row_index]
                .astype(str)
                .str.strip()
                .str.replace(r"\s+%", "%", regex=True)
                .str.replace(r"\s+", " ", regex=True)
                .str.upper()
            )
            df.columns = header

            ano_ref, mes_ref = int(filename.split("_")[0]), int(filename.split("_")[1])
            df["ANO_REF"], df["MES_REF"] = ano_ref, mes_ref

            cols_to_move = ["ANO_REF", "MES_REF"]
            df = df[cols_to_move + [c for c in df.columns if c not in cols_to_move]]

            output_name = output_name or f"ANVISA_LIMPO_{ano_ref}_{mes_ref:02d}.csv"
            os.makedirs(target_folder, exist_ok=True)
            output_path = output_path or os.path.join(target_folder, output_name)
            df.to_csv(output_path, sep=";", index=False, encoding="utf-8")

            engine_msg = f" (engine: {engine_used})" if ext == ".xls" else ""
            return f"OK: {filename} -> {output_name}{engine_msg}"
        except Exception as exc:
            return f"ERRO: {file_path} -> {exc}"

    workers_limpeza = min(MAX_CLEANING_THREADS * 2, 12)
    logging.info(f"Iniciando limpeza de {len(all_files)} arquivos com {workers_limpeza} threads...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers_limpeza) as exe:
        resultados = list(
            tqdm(
                exe.map(process_single_file, all_files),
                total=len(all_files),
                desc="Limpando arquivos",
                ncols=100,
            )
        )

    logging.info("--- Resultados da Limpeza ---")
    for r in resultados:
        logging.info(f" -> {r}")

def consolidate_cleaned_files(source_folder, output_file):
    """Consolida todos os CSVs limpos em um unico arquivo."""
    csv_files = sorted(glob.glob(os.path.join(source_folder, "*.csv")))
    if not csv_files:
        logging.warning("Nenhum arquivo CSV limpo encontrado para consolidar.")
        return None

    colunas_para_manter = [
        "ANO_REF", "MES_REF", "PRINCIPIO ATIVO", "LABORATORIO", "CODIGO GGREM", "REGISTRO",
        "EAN 1", "EAN 2", "EAN 3", "PRODUTO", "APRESENTACAO", "CLASSE TERAPEUTICA",
        "TIPO DE PRODUTO (STATUS DO PRODUTO)", "REGIME DE PRECO",
        "PF 0%", "PF 18%", "PF 20%", "PMVG 0%", "PMVG 18%", "PMVG 20%", "ICMS 0%", "CAP",
    ]
    variantes_principio = ["PRINCIPIO ATIVO", "PRINCÍPIO ATIVO", "SUBSTANCIA", "SUBSTÂNCIA"]

    dfs = []
    for file in tqdm(csv_files, desc="Lendo CSVs limpos", ncols=100):
        try:
            df = pd.read_csv(file, sep=";", dtype=str, low_memory=False, engine="c")
            df.columns = df.columns.str.strip().str.upper()

            col_principio = next((c for c in df.columns if c in variantes_principio), None)
            if col_principio and col_principio != "PRINCIPIO ATIVO":
                df.rename(columns={col_principio: "PRINCIPIO ATIVO"}, inplace=True)
            elif "PRINCIPIO ATIVO" not in df.columns:
                df["PRINCIPIO ATIVO"] = None

            colunas_existentes = [c for c in colunas_para_manter if c in df.columns]
            dfs.append(df[colunas_existentes])
        except Exception as exc:
            logging.error(f"Erro ao ler {file}: {exc}")

    if not dfs:
        logging.error("Nenhum DataFrame valido foi carregado para consolidacao.")
        return None

    logging.info("Concatenando bases...")
    df_consolidado = pd.concat(dfs, ignore_index=True, sort=False).dropna(how="all")

    if "PRODUTO" in df_consolidado.columns and "PRINCIPIO ATIVO" in df_consolidado.columns:
        df_consolidado = df_consolidado.dropna(subset=["PRODUTO", "PRINCIPIO ATIVO"])

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    df_consolidado.to_csv(output_file, sep=";", index=False, encoding="utf-8")
    logging.info(f"Consolidacao concluida. Arquivo salvo em: {os.path.abspath(output_file)}")
    return df_consolidado


In [ ]:
# Parametros de execucao
FORCE_REFRESH = False
SKIP_DOWNLOAD = False

# Defina overrides para testar janelas especificas.
# Se ficar como None, usa o periodo padrao de config_anvisa.py
ANO_INICIO_OVERRIDE = None
MES_INICIO_OVERRIDE = None
ANO_FIM_OVERRIDE = None
MES_FIM_OVERRIDE = None

ano_inicio = ANO_INICIO_OVERRIDE if ANO_INICIO_OVERRIDE is not None else CONF_ANO_INICIO
mes_inicio = MES_INICIO_OVERRIDE if MES_INICIO_OVERRIDE is not None else CONF_MES_INICIO
ano_fim = ANO_FIM_OVERRIDE if ANO_FIM_OVERRIDE is not None else CONF_ANO_FIM
mes_fim = MES_FIM_OVERRIDE if MES_FIM_OVERRIDE is not None else CONF_MES_FIM

data_inicio = datetime(ano_inicio, mes_inicio, 1)
data_fim = datetime(ano_fim, mes_fim, 1)

print(f"Periodo efetivo: {mes_inicio:02d}/{ano_inicio} ate {mes_fim:02d}/{ano_fim}")
print(f"FORCE_REFRESH={FORCE_REFRESH}")
print(f"SKIP_DOWNLOAD={SKIP_DOWNLOAD}")
print(f"Downloads brutos: {PASTA_DOWNLOADS_BRUTOS}")
print(f"Arquivos limpos: {PASTA_ARQUIVOS_LIMPOS}")
print(f"Consolidado bruto: {ARQUIVO_CONSOLIDADO_TEMP}")


Periodo efetivo: 09/2022 ate 11/2022
FORCE_REFRESH=False
SKIP_DOWNLOAD=False
Downloads brutos: data/raw
Arquivos limpos: data/processed
Consolidado bruto: data/processed/anvisa/anvisa_pmvg_consolidado_temp.csv


In [4]:
def run_stage1_download_notebook(
    force_refresh: bool = False,
    skip_download: bool = False,
    ano_inicio_exec: int | None = None,
    mes_inicio_exec: int | None = None,
    ano_fim_exec: int | None = None,
    mes_fim_exec: int | None = None,
):
    """Executa o pipeline 1.0 completo, de forma autocontida no notebook."""
    ano_inicio_ref = CONF_ANO_INICIO if ano_inicio_exec is None else ano_inicio_exec
    mes_inicio_ref = CONF_MES_INICIO if mes_inicio_exec is None else mes_inicio_exec
    ano_fim_ref = CONF_ANO_FIM if ano_fim_exec is None else ano_fim_exec
    mes_fim_ref = CONF_MES_FIM if mes_fim_exec is None else mes_fim_exec

    if not logging.getLogger().handlers:
        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s - %(levelname)s - %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        )

    logging.info(f"Periodo de coleta: {mes_inicio_ref:02d}/{ano_inicio_ref} ate {mes_fim_ref:02d}/{ano_fim_ref}")
    logging.info("=" * 80)
    logging.info("PIPELINE 1.0 - DOWNLOAD E CONSOLIDACAO BRUTA DA ANVISA (NOTEBOOK)")
    logging.info("=" * 80)

    # 1) Limpeza inicial opcional (refresh completo)
    if force_refresh and os.path.exists(PASTA_DOWNLOADS_BRUTOS):
        shutil.rmtree(PASTA_DOWNLOADS_BRUTOS)
        logging.info(f"Pasta antiga '{PASTA_DOWNLOADS_BRUTOS}' removida (force_refresh).")

    os.makedirs(PASTA_DOWNLOADS_BRUTOS, exist_ok=True)
    os.makedirs(PASTA_ARQUIVOS_LIMPOS, exist_ok=True)
    logging.info("Estrutura de pastas criada.")

    data_inicio_ref = datetime(ano_inicio_ref, mes_inicio_ref, 1)
    data_fim_ref = datetime(ano_fim_ref, mes_fim_ref, 1)

    df_links = None
    df_to_download = None

    if not skip_download:
        # 2) Raspagem de links
        df_links = scrape_anvisa_links()

        # 3) Filtragem por periodo
        df_to_download = df_links[df_links.apply(
            lambda row: data_inicio_ref <= datetime(int(row["ano"]), int(row["mes"]), 1) <= data_fim_ref,
            axis=1,
        )].copy()

        # 4) Download dos arquivos
        if df_to_download.empty:
            logging.warning("Nenhum link encontrado no periodo selecionado.")
        else:
            download_files(df_to_download)
    else:
        logging.info("Skip download habilitado: reutilizando base local.")

    # 5) Limpeza dos arquivos
    clean_downloaded_files(PASTA_DOWNLOADS_BRUTOS, PASTA_ARQUIVOS_LIMPOS)

    # 6) Consolidacao bruta
    df_consolidado = consolidate_cleaned_files(PASTA_ARQUIVOS_LIMPOS, ARQUIVO_CONSOLIDADO_TEMP)
    if df_consolidado is None:
        raise RuntimeError("A consolidacao falhou. Verifique os logs da execucao.")

    resumo_execucao = {
        "periodo_inicio": data_inicio_ref.strftime("%Y-%m"),
        "periodo_fim": data_fim_ref.strftime("%Y-%m"),
        "force_refresh": bool(force_refresh),
        "skip_download": bool(skip_download),
        "links_capturados": 0 if df_links is None else int(len(df_links)),
        "links_no_periodo": 0 if df_to_download is None else int(len(df_to_download)),
        "linhas_consolidadas": int(len(df_consolidado)),
        "arquivo_consolidado": str(Path(ARQUIVO_CONSOLIDADO_TEMP).resolve()),
    }

    logging.info("[OK] Pipeline 1.0 concluido no notebook.")
    logging.info(f"Arquivo consolidado bruto: {Path(ARQUIVO_CONSOLIDADO_TEMP).resolve()}")
    logging.info(f"Tamanho: {len(df_consolidado):,} linhas")

    return resumo_execucao, df_links, df_to_download, df_consolidado

resumo_execucao, df_links, df_to_download, df_consolidado = run_stage1_download_notebook(
    force_refresh=FORCE_REFRESH,
    skip_download=SKIP_DOWNLOAD,
    ano_inicio_exec=ANO_INICIO_OVERRIDE,
    mes_inicio_exec=MES_INICIO_OVERRIDE,
    ano_fim_exec=ANO_FIM_OVERRIDE,
    mes_fim_exec=MES_FIM_OVERRIDE,
)

display(pd.DataFrame([resumo_execucao]))
display(df_consolidado.head(5))


2026-03-29 12:29:43 - INFO - Periodo de coleta: 09/2022 ate 11/2022
2026-03-29 12:29:43 - INFO - ================================================================================
2026-03-29 12:29:43 - INFO - PIPELINE 1.0 - DOWNLOAD E CONSOLIDACAO BRUTA DA ANVISA (NOTEBOOK)
2026-03-29 12:29:43 - INFO - ================================================================================
2026-03-29 12:29:43 - INFO - Estrutura de pastas criada.
2026-03-29 12:29:43 - INFO - Acessando https://www.gov.br/anvisa/pt-br/assuntos/medicamentos/cmed/precos/anos-anteriores/anos-anteriores para extrair links...
2026-03-29 12:29:44 - INFO - Total de links capturados: 189
2026-03-29 12:29:44 - INFO - Arquivos ja disponiveis localmente (skip): 3
2026-03-29 12:29:44 - INFO - Nenhum arquivo novo para download.
2026-03-29 12:29:44 - INFO - Iniciando limpeza de 77 arquivos com 12 threads...
Limpando arquivos: 100%|██████████████████████████████████████████| 77/77 [00:00<00:00, 7004.30it/s]
2026-03-29 12:29:44 - 

,periodo_inicio,periodo_fim,force_refresh,skip_download,links_capturados,links_no_periodo,linhas_consolidadas,arquivo_consolidado
0,2022-09,2022-11,False,False,189,3,1913203,C:\Users\luciano\Desktop\Works\Pipeline_Anvisa...


,ANO_REF,MES_REF,PRINCIPIO ATIVO,REGISTRO,EAN 1,EAN 2,EAN 3,PRODUTO,TIPO DE PRODUTO (STATUS DO PRODUTO),PF 0%,PF 18%,PF 20%,PMVG 0%,PMVG 18%,PMVG 20%,ICMS 0%,CAP,LABORATORIO,APRESENTACAO,CLASSE TERAPEUTICA
0,2020,1,ABATACEPTE,1018003900078,7896016808197,NaN,NaN,ORENCIA,Biológicos,"4760,29","5805,23","5950,36","3800,62",NaN,NaN,Não,Não,NaN,NaN,NaN
1,2020,1,ABATACEPTE,1018003900061,7896016808180,NaN,NaN,ORENCIA,Biológicos,"1190,06","1451,29","1487,57","950,14","1158,71","1187,68",Não,Não,NaN,NaN,NaN
2,2020,1,ABATACEPTE,1018003900051,7896016807473,NaN,NaN,ORENCIA,Biológicos,"4760,29","5805,23","5950,36","3800,62","4634,90","4750,77",Não,Não,NaN,NaN,NaN
3,2020,1,ABATACEPTE,1018003900043,7896016807459,NaN,NaN,ORENCIA,Biológicos,"1190,06","1451,29","1487,57","950,14","1158,71","1187,68",Não,Não,NaN,NaN,NaN
4,2020,1,ABATACEPTE,1018003900035,7896016807466,NaN,NaN,ORENCIA,Biológicos,"4760,29","5805,23","5950,36","3800,62","4634,90","4750,77",Não,Não,NaN,NaN,NaN


## Proximo passo

Este notebook cobre a etapa 1.0 (download e consolidacao bruta) com execucao unica por celula.
Para seguir no fluxo completo da base ANVISA (1.0 + 1.5 + 2B), execute:

```bash
python scripts/run_anvisa_completo.py
```

Se preferir por notebooks, continue no notebook da etapa 1.5/2B.
